In [ ]:
from random import seed

# 前馈神经网络
import torch.nn as nn
from sympy.printing.pytorch import torch

from chapter02_transformer架构.注意力机制 import MultiHeadAttention


class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dropout: float):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.relu(self.w1(x)))




In [ ]:
# 归一化层
class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))

        self.eps = eps

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        std = x.std(-1, keepdim=True)
        return self.a_2 * (x - mean) / (std + self.eps) + self.b_2



In [ ]:
# 残差连接
# h = x + self.attention.forward(self.attention_norm(x))
# 经过前馈神经⽹络
# out = h + self.feed_forward.forward(self.fnn_norm(h))


In [ ]:
# Encoder

class EncoderLayer(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.attention_norm = LayerNorm(args.n_embd)

        self.attention = MultiHeadAttention(args, is_causal=False)
        self.fnn_norm = LayerNorm(args.n_embd)
        self.feed_forward = NLP(args)

    def forward(self, x):
        norm_x = self.attention_norm(x)

        h = x + self.attention.forward(norm_x, norm_x)

        out = h + self.feed_forward.forward(self.fnn_norm(h))

        return out


class Encoder(nn.Module):
    def __init__(self, args):
        super(Encoder, self).__init__()

        self.layers = nn.ModuleList([EncoderLayer(args) for _ in range(args.n_layer)])
        self.norm = LayerNorm(args.n_embd)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)



In [ ]:
#  Decoder

class DecoderLayer(nn.Module):
    def __init__(self, args):
        super().__init__()
        self.attention_norm_1 = LayerNorm(args.n_embd)
        self.mask_attention = MultiHeadAttention(args, is_causal=True)
        self.attention_norm_2 = LayerNorm(args.n_embd)
        self.attention = MultiHeadAttention(args, is_causal=False)
        self.fn_norm = LayerNorm(args.n_embd)
        self.feed_forward = NLP(args)

    def forward(self, x, enc_out):
        norm_x = self.attention.norm_1(x)
        x = x + self.maxk_attention.forward(x, norm_x, norm_x)
        norm_x = self.attention_norm_2(x)
        h = x + self.attention.forward(norm_x, enc_out, enc_out)
        out = h + self.feed_forward.forward(self.ffn_norm(h))
        return out


class Decoder(nn.Module):
    def __init__(self, args):
        super(Decoder, self).__init__()
        self.layers = [DecoderLayer(args) for _ in range(args.n_layer)]
        self.norm = LayerNorm(args.n_embd)

    def forward(self, x, enc_out):
        for layer in self.layers:
            x = layer(x, enc_out)
        return self.norm(x)


